# Computational Theory Problems

This notebook works through a series of problems based on the Secure Hash Standard published by NIST as FIPS PUB 180-4. 

## Problem 1: Representing SHA-256 Data

SHA-256 works on fixed-size binary data. Before implementing the algorithm, this section chooses a Python representation for each kind of data it uses, based on the definitions in FIPS 180-4.

The choice matters because Python's built-in `int` has no fixed size. SHA-256 depends on values that are exactly 32 bits wide and that wrap around when they overflow ([FIPS 180-4, Section 3.2](https://doi.org/10.6028/NIST.FIPS.180-4)). The wrong representation produces a wrong hash without raising any error.

For each kind of data, this section:

- states what the standard requires
- explains why the chosen representation is appropiate
- demonstrates the chosen representation with an example

All multi-byte values are read and written in big-endian order (see Section 4).



| Data | Size | Representation |
| --- | --- | --- |
| Word | 32 bits | `np.uint32` |
| Sequence of words | $n \times 32$ bits | 1-D NumPy array of `np.uint32` |
| Input message | Any whole number of bytes | `bytes` |
| Message block | 512 bits | NumPy array of 16 `np.uint32` words |


Each choice is explained and demonstrated below.

### Imports

[NumPy](https://numpy.org/doc/stable/user/basics.types.html) provides fixed-width integer types such as `uint32`. The standard library module `sys` reports the machine's native byte order.

In [3]:
# Standard library: reports this machine's native byte order.
import sys

# NumPy: provides fixed-width unsigned integers such as uint32.
import numpy as np

### 1. 32-bit Words

FIPS 180-4 ([Section 2.1](https://doi.org/10.6028/NIST.FIPS.180-4)) defines a *word* as a group of $w$ bits, with $w = 32$ for SHA-256. Every SHA-256 operation works on words.

**Choice:** NumPy's `np.uint32`, an unsigned (non-negative) integer that is always exactly 32 bits wide.

**Why it is appropriate:**

- It can't hold more than 32 bits which matches the defintion of a word
- Addition wraps around after $2^{32} - 1$, as required by Section 3.2
- A lift shift removes any bit moved after the 32nd position

Each of these properties is checked below.


In [4]:
# Create a single 32-bit word holding the largest possible value.
word = np.uint32(0xFFFFFFFF)

print(word)
print(type(word))
print(hex(word))

4294967295
<class 'numpy.uint32'>
0xffffffff


**Exactly 32 bits.**

 A 32-bit unsigned integer ranges from $0$ to $2^{32} - 1$. Instead of entering these limits by hand, [`np.iinfo`](https://numpy.org/doc/stable/reference/generated/numpy.iinfo.html) reads them directly from NumPy.

In [5]:
# Show the smallest and largest values a uint32 can hold.
limits = np.iinfo(np.uint32)

print(limits.min, limits.max)
print(hex(limits.max))

0 4294967295
0xffffffff


**Addition wraps modulo $2^{32}$.** 

Adding 1 to the largest `uint32` value gives 0. The same sum with a Python `int` is included for contrast: it continues to 4294967296, which is not a valid 32-bit word.

An array is used here rather than a single value. Adding two single `np.uint32` values also gives 0, but NumPy prints an overflow warning as well. An array wraps without the warning.

In [6]:
# A uint32 array wraps round to 0 when it overflows.
words = np.array([0xFFFFFFFF], dtype=np.uint32)
print(words + np.uint32(1))

# A Python int keeps growing past 32 bits.
print(0xFFFFFFFF + 1)

[0]
4294967296


**Shifting stays within 32 bits.** 

The value `0x80000001` has its highest and lowest bits set. Shifting it left by one moves the highest bit outside the 32-bit range. A `uint32` removes that bit and gives `0x2`, while a Python `int` keeps it and becomes 33 bits long.

In [7]:
# A value with its highest and lowest bits set.
x = np.uint32(0x80000001)

# Shift left by one: the highest bit falls off a uint32.
print(hex(x << 1))

# The same shift on a Python int keeps the extra bit.
print(hex(0x80000001 << 1))

0x2
0x100000002


### 2. Sequences of 32-bit Words

SHA-256 uses several fixed-length sequences of words

8 words for the hash value ([FIPS 180-4, Section 5.3.3](https://doi.org/10.6028/NIST.FIPS.180-4)), 
16 for each message block (Section 5.2.1), and 64 each for the round constants (Section 4.2.2) and the message schedule (Section 6.2.2).

**Choice:** a one-dimensional [NumPy array](https://numpy.org/doc/stable/reference/arrays.ndarray.html) with `dtype=np.uint32`.

**Why it is appropriate:**

- Every element is stored as a `uint32`, so each one keeps the 32-bit behaviour from Section 1
- An operation applied to the whole array is applied to every element at once and each result still wraps at 32 bits
- The words are kept in order and can be accessed by position which matches the subscript notation in the standard like $H_0$ to $H_7$

**Fixed type and size.** 

The eight initial hash values $H^{(0)}$ are stored below. The array reports its type, its length and its total size in bytes. Eight words of 4 bytes each give 32 bytes which is 256 bits.

In [8]:
# The eight initial hash values H(0) from FIPS 180-4, Section 5.3.3.
initial_hash = np.array(
    [0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,
     0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19],
    dtype=np.uint32,
)

print(initial_hash.dtype, initial_hash.shape, initial_hash.nbytes)

# A single element keeps the uint32 type.
print(type(initial_hash[0]))

uint32 (8,) 32
<class 'numpy.uint32'>


**Operations stay within 32 bits.**

 Here the array is added to itself so every value gets doubled in one go. Each answer wraps around at $2^{32}$ so they all stay as proper 32-bit words. The same values in a normal Python `list` are shown underneath for comparason. Four of them go over 32 bits, which you can see from the extra `1` at the front.

In [9]:
# Every element is added at once, and each sum wraps at 2**32.
print([hex(word) for word in initial_hash + initial_hash])

# The same values in a Python list are plain ints and do not wrap.
hash_list = [0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,
             0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19]
print([hex(word + word) for word in hash_list])

['0xd413ccce', '0x76cf5d0a', '0x78dde6e4', '0x4a9fea74', '0xa21ca4fe', '0x360ad118', '0x3f07b356', '0xb7c19a32']
['0xd413ccce', '0x176cf5d0a', '0x78dde6e4', '0x14a9fea74', '0xa21ca4fe', '0x1360ad118', '0x3f07b356', '0xb7c19a32']


### 3. Input Messages

The standard treats the message as a sequence of bits with a length of $\ell$ bits ([FIPS 180-4, Section 5.1.1](https://doi.org/10.6028/NIST.FIPS.180-4)). In practice, every message is made up of whole bytes, so $\ell$ is always a multiple of 8.

**Choice:** Python's built-in [`bytes`](https://docs.python.org/3/library/stdtypes.html#bytes-objects) type.

**Why it is appropriate:**

- Each element is one byte, a whole number from 0 to 255, so the message is stored exactly as SHA-256 reads it.
- The length in bits, $\ell$, is simply `len(message) * 8`. This value is needed for padding in Problem 4.
- A `bytes` object cannot be changed after it is created, so the message cannot be altered by mistake while it is being hashed.
- Python's [`hashlib.sha256`](https://docs.python.org/3/library/hashlib.html) also takes `bytes`, so the same input can be passed to both when testing in Problem 6.

**Text must be encoded first.** 

SHA-256 works on bytes, not text, so a string is converted with [`str.encode`](https://docs.python.org/3/library/stdtypes.html#str.encode) using UTF-8 (the standard way of storing text as bytes). The message `"abc"` is used here because it is one of the standard SHA-256 test messages published by NIST.

In [10]:
# Convert the text "abc" into bytes using UTF-8.
message = "abc".encode("utf-8")

print(message)
print(type(message))

# Each byte is a number from 0 to 255.
print(list(message))

# The message length in bits, written as l in the standard.
print(len(message) * 8, "bits")

b'abc'
<class 'bytes'>
[97, 98, 99]
24 bits


**Characters and bytes are not the same.** 

In UTF-8, a basic English letter takes 1 byte, but other characters can take 2 to 4 bytes. This means the message length has to come from the encoded bytes, not from the number of characters in the text. This matters for the non-ASCII test messages in Problem 6.

In [11]:
# Compare the number of characters with the number of bytes after encoding.
for text in ["abc", "café", "€"]:
    encoded = text.encode("utf-8")
    print(f"{text!r} -> characters: {len(text)}, bytes: {len(encoded)}, {encoded}")

'abc' -> characters: 3, bytes: 3, b'abc'
'café' -> characters: 4, bytes: 5, b'caf\xc3\xa9'
'€' -> characters: 1, bytes: 3, b'\xe2\x82\xac'


**The message cannot be changed by mistake.** 

`bytes` objects are immutable (cannot be changed after they are created). Trying to change a byte raises an error instead of quietly altering the message.

In [12]:
# Try to replace the first byte; bytes objects do not allow this.
try:
    message[0] = 120
except TypeError as error:
    print("TypeError:", error)

TypeError: 'bytes' object does not support item assignment


### 4. Big-Endian Integers

When a number is stored in more than one byte, the bytes can be written in two orders. *Big-endian* puts the most significant byte (the "biggest" part of the number) first, the same way numbers are written on paper. *Little-endian* puts the least significant byte first. The terms come from Danny Cohen's 1980 note [*On Holy Wars and a Plea for Peace*](https://www.rfc-editor.org/ien/ien137.txt).

The Secure Hash Standard uses the big-endian convention throughout ([FIPS 180-4, Section 3.1](https://doi.org/10.6028/NIST.FIPS.180-4)). This affects two later steps: reading each group of 4 message bytes as one word (Problem 4), and writing the message length into the padding (Problem 4).

**Choice:** always state the byte order explicitly. Python integers use [`int.to_bytes` and `int.from_bytes`](https://docs.python.org/3/library/stdtypes.html#int.to_bytes) with `"big"`. NumPy arrays use the dtype `">u4"`, where `>` means big-endian, `u` means unsigned and `4` means 4 bytes ([NumPy byte order](https://numpy.org/doc/stable/user/basics.byteswapping.html)).

**Why it is appropriate:**

- Most modern processors are little-endian, so leaving the order to the machine gives the bytes in the wrong order for SHA-256.
- Naming the order makes the code give the same result on every machine.
- No error is raised when the order is wrong, so it has to be correct by design rather than caught later.

**This machine's byte order.** 

[`sys.byteorder`](https://docs.python.org/3/library/sys.html#sys.byteorder) reports the order the machine uses by default.

In [13]:
# Show the native byte order of the machine running this notebook.
print(sys.byteorder)

little


**One number, two byte orders.** 

The number `0x12345678` is made of the four bytes `12`, `34`, `56` and `78`. Big-endian keeps them in the order they are written. Little-endian reverses them.

In [14]:
# Write the same 32-bit number as 4 bytes in each order.
value = 0x12345678

# hex(" ") puts a space between each byte so the order is easy to see.
print("big:   ", value.to_bytes(4, "big").hex(" "))
print("little:", value.to_bytes(4, "little").hex(" "))

big:    12 34 56 78
little: 78 56 34 12


**Reading bytes with the wrong order.** 

The same four bytes give two different numbers depending on the order used to read them. NumPy's plain `np.uint32` uses the machine's own order, which on a little-endian machine gives the wrong word for SHA-256. The `">u4"` dtype forces big-endian and gives the correct word on any machine.

In [15]:
# Four bytes as they would appear in a message.
raw = bytes([0x12, 0x34, 0x56, 0x78])

# Python integers: the order is named in the call.
print("int, big:      ", hex(int.from_bytes(raw, "big")))
print("int, little:   ", hex(int.from_bytes(raw, "little")))

# NumPy: ">u4" is big-endian; plain np.uint32 uses the machine's order.
print("numpy, >u4:    ", hex(np.frombuffer(raw, dtype=">u4")[0]))
print("numpy, native: ", hex(np.frombuffer(raw, dtype=np.uint32)[0]))

int, big:       0x12345678
int, little:    0x78563412
numpy, >u4:     0x12345678
numpy, native:  0x78563412


**The message length in the padding.**

 SHA-256 ends the padded message with the original message length in bits, written as a 64-bit (8-byte) big-endian integer ([FIPS 180-4, Section 5.1.1](https://doi.org/10.6028/NIST.FIPS.180-4)). For the 3-byte message `"abc"`, the length is 24 bits, which is `0x18` in hexadecimal.

In [16]:
# The length of "abc" in bits, as an 8-byte big-endian integer.
message_length = len(b"abc") * 8
print(message_length.to_bytes(8, "big").hex(" "))

00 00 00 00 00 00 00 18


### 5. 512-bit Message Blocks

After padding, the message is split into blocks of 512 bits ([FIPS 180-4, Section 5.2.1](https://doi.org/10.6028/NIST.FIPS.180-4)). Each block is then read as sixteen 32-bit words, written $M_0^{(i)}$ to $M_{15}^{(i)}$ for block $i$. These sixteen words become the first sixteen entries of the message schedule in the compression function (Section 6.2.2).

$$512 \text{ bits} = 64 \text{ bytes} = 16 \text{ words} \times 32 \text{ bits}$$

**Choice:** a NumPy array of 16 `np.uint32` words, made from 64 bytes with [`np.frombuffer`](https://numpy.org/doc/stable/reference/generated/numpy.frombuffer.html) using the big-endian dtype `">u4"`, then converted with `.astype(np.uint32)`.

**Why it is appropriate:**

- `np.frombuffer` turns the bytes into words in one step, without a Python loop.
- `">u4"` reads each group of 4 bytes big-endian, as required by Section 3.1 (see Section 4 above).
- `.astype(np.uint32)` converts the words into the machine's own `uint32` type, so they behave the same as every other word in the notebook.
- The result is the same kind of array as in Section 2, so the functions in later problems can use it directly.

**Building one block.** 

The padded form of `"abc"` fits in a single block: the 3 message bytes, a `0x80` byte (a `1` bit followed by seven `0` bits), 52 zero bytes, and the message length of 24 bits as an 8-byte big-endian integer. The padding rules are implemented properly in Problem 4. Here the block is built by hand to give a real example.

In [17]:
# The padded "abc" message: message, 0x80, zeros, then the length in bits.
padded = b"abc" + b"\x80" + bytes(52) + (24).to_bytes(8, "big")

print(len(padded), "bytes,", len(padded) * 8, "bits")

64 bytes, 512 bits


**Reading the block as 16 words.** 

`np.frombuffer` with `">u4"` gives 16 big-endian words. The `.astype(np.uint32)` step converts them to the machine's own `uint32` type. It also makes a copy that can be changed, since an array made by `np.frombuffer` is read-only.

In [18]:
# Read each group of 4 bytes as one big-endian 32-bit word.
raw_words = np.frombuffer(padded, dtype=">u4")
print(raw_words.dtype, raw_words.shape, "writeable:", raw_words.flags.writeable)

# Convert to the machine's own uint32 type, as used everywhere else.
block = raw_words.astype(np.uint32)
print(block.dtype, block.shape, "writeable:", block.flags.writeable)

>u4 (16,) writeable: False
uint32 (16,) writeable: True


**Checking the words.** 

The first word holds `"abc"` and the `0x80` byte, and the last word holds the length, 24 (`0x18`). These match the `"abc"` example published by NIST on its [example values page](https://csrc.nist.gov/projects/cryptographic-standards-and-guidelines/example-values).

In [19]:
# Print each word as 8 hex digits, labelled M0 to M15 as in the standard.
for index, word in enumerate(block):
    print(f"M{index:<2} = {int(word):08x}")

M0  = 61626380
M1  = 00000000
M2  = 00000000
M3  = 00000000
M4  = 00000000
M5  = 00000000
M6  = 00000000
M7  = 00000000
M8  = 00000000
M9  = 00000000
M10 = 00000000
M11 = 00000000
M12 = 00000000
M13 = 00000000
M14 = 00000000
M15 = 00000018


**Longer messages give more blocks.** 

A padded message is always a multiple of 64 bytes. [`reshape(-1, 16)`](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html) arranges the words into rows of 16, one row per block, where `-1` tells NumPy to work out the number of rows itself. This example uses 128 bytes with the values 0 to 127, which gives two blocks.

In [20]:
# 128 example bytes (0x00 to 0x7F): enough for exactly two blocks.
two_blocks = np.frombuffer(bytes(range(128)), dtype=">u4").astype(np.uint32)

# One row of 16 words per block.
two_blocks = two_blocks.reshape(-1, 16)
print(two_blocks.shape)

# The first word of each block.
print(f"{int(two_blocks[0][0]):08x} {int(two_blocks[1][0]):08x}")

(2, 16)
00010203 40414243
